# Temporal Comparison — Summary
RMSD bar chart grouped by joint and model pair.
Edit the **CONFIGURATION** cell paths before running.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ezc3d

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  ← edit paths here
# ══════════════════════════════════════════════════════════════════════════════

# Each joint entry: display name, variable name, Euler component, and 3 C3D
# paths for [Complete, With C7, Without C7] — for ONE side only (e.g. Left).
# Repeat for Right if needed by duplicating entries with different paths.

JOINTS = [
    {
        "name":      "Shoulder Flex/Ext",
        "variable":  "Humerothoracic_ZXY_Op1",
        "component": 0,
        "side":      "Left",
        "paths": [
            r"C:\Projects\data_organized\grab 5_c3d\De pie_c3d\Pie_Complete_Shoulder_Flex_Ext_Left_Cont01.c3d",
            r"C:\Projects\data_organized\grab 5_c3d\De pie_c3d\Pie_C7_Shoulder_Flex_Ext_Left_Cont01.c3d",
            r"C:\Projects\data_organized\grab 5_c3d\De pie_c3d\Pie_NOT_C7_Shoulder_Flex_Ext_Left_Cont01.c3d",
        ],
    },
    {
        "name":      "Shoulder Abd/Add",
        "variable":  "Humerothoracic_XZY_Op1",
        "component": 0,
        "side":      "Left",
        "paths": [
            r"C:\Projects\c3d_data_organized\grab_5\Pie_Complete_Shoulder_Abd_Add_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_C7_Shoulder_Abd_Add_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_NOT_C7_Shoulder_Abd_Add_Left_Cont01.c3d",
        ],
    },
    {
        "name":      "Shoulder Int/Ext Rot",
        "variable":  "Humerothoracic_XZY_Op2",
        "component": 2,
        "side":      "Left",
        "paths": [
            r"C:\Projects\c3d_data_organized\grab_5\Pie_Complete_Shoulder_Rotac_Int_Ext_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_C7_Shoulder_Rotac_Int_Ext_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_NOT_C7_Shoulder_Rotac_Int_Ext_Left_Cont01.c3d",
        ],
    },
    {
        "name":      "Elbow Flex/Ext",
        "variable":  "Elbow_Op1",
        "component": 0,
        "side":      "Left",
        "paths": [
            r"C:\Projects\c3d_data_organized\grab_5\Pie_Complete_Elbow_Flex_Ext_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_C7_Elbow_Flex_Ext_Left_Cont01.c3d",
            r"C:\Projects\c3d_data_organized\grab_5\Pie_NOT_C7_Elbow_Flex_Ext_Left_Cont01.c3d",
        ],
    },
]

PAIR_LABELS = ["Complete \u2212 With C7", "Complete \u2212 Without C7", "With C7 \u2212 Without C7"]
PAIR_COLORS = ["#2E86C1", "#27AE60", "#E67E22"]

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def read_c3d(filepath):
    filepath = os.path.normpath(os.path.abspath(filepath))
    c = ezc3d.c3d(filepath)
    raw_labels = c["parameters"]["POINT"]["LABELS"]["value"]
    point_data = c["data"]["points"]
    n_labels   = min(len(raw_labels), point_data.shape[1])
    try:
        rate_val   = c["parameters"]["POINT"]["RATE"]["value"]
        frame_rate = int(float(rate_val[0]) if hasattr(rate_val, "__len__") else float(rate_val))
    except Exception:
        frame_rate = int(c["header"]["frame_rate"])
    clean_labels = [
        lbl.split(":")[-1].strip() if ":" in lbl else lbl.strip()
        for lbl in raw_labels[:n_labels]
    ]
    model_outputs = {lbl: point_data[:3, i, :].copy() for i, lbl in enumerate(clean_labels)}
    return {"frame_rate": frame_rate, "model_outputs": model_outputs}


def extract_signal(path, variable, component, side):
    c3d   = read_c3d(path)
    label = variable if side == "—" else side + variable
    return c3d["model_outputs"][label][component, :].astype(float)


def rmsd(a, b):
    n = min(len(a), len(b))
    return float(np.sqrt(np.mean((a[:n] - b[:n])**2)))

In [ ]:
# ── Compute RMSD for every joint × pair ──────────────────────────────────────

results = []   # list of (joint_name, [rmsd_c7, rmsd_no_c7, rmsd_c7_no_c7])

for j in JOINTS:
    sig_complete  = extract_signal(j["paths"][0], j["variable"], j["component"], j["side"])
    sig_c7        = extract_signal(j["paths"][1], j["variable"], j["component"], j["side"])
    sig_no_c7     = extract_signal(j["paths"][2], j["variable"], j["component"], j["side"])

    r_c7       = rmsd(sig_complete, sig_c7)
    r_no_c7    = rmsd(sig_complete, sig_no_c7)
    r_c7_no_c7 = rmsd(sig_c7, sig_no_c7)

    results.append((j["name"], [r_c7, r_no_c7, r_c7_no_c7]))
    print(f"{j['name']:25s}  RMSD: {r_c7:.2f} / {r_no_c7:.2f} / {r_c7_no_c7:.2f} \u00b0")

In [ ]:
# ── RMSD bar chart grouped by joint ──────────────────────────────────────────

n_joints = len(results)
n_pairs  = 3
x        = np.arange(n_joints)
width    = 0.25
offsets  = [-width, 0, width]

fig, ax = plt.subplots(figsize=(max(8, n_joints * 2), 5))
fig.suptitle("RMSD Summary — All Joints and Model Pairs", fontsize=13, fontweight="bold")

for p_idx, (label, color, offset) in enumerate(zip(PAIR_LABELS, PAIR_COLORS, offsets)):
    values = [results[j][1][p_idx] for j in range(n_joints)]
    bars   = ax.bar(x + offset, values, width, color=color, alpha=0.85, label=label)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([r[0] for r in results], fontsize=9)
ax.set_ylabel("RMSD (\u00b0)", fontsize=10)
ax.legend(fontsize=9, framealpha=0.7)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.grid(True, linestyle="--", alpha=0.35, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()